In [ ]:
from odps_client import get_odps_sql_result_as_df
from datetime import datetime, timedelta

daily_high_search_volume_theshold = 470 / 14
daily_low_search_volume_theshold = 2

last_n_days = 60
ds_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
last_n_days_ago = (datetime.now() - timedelta(days=last_n_days)).strftime("%Y%m%d")

top_query = f"""
SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.5) AS p50_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.75) AS p75_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.9) AS p90_click_index
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > {last_n_days*daily_high_search_volume_theshold} THEN '高频搜索词'
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) <= {last_n_days*daily_low_search_volume_theshold} THEN '低频搜索词'
            ELSE '中频搜索词'
         END AS 搜索频次标签
        ,COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) as ctr_uv
        ,COUNT(distinct ds) as 有搜索天数
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > 0.25 THEN '高点击率词'
            ELSE '低点击率词'
         END AS 点击率标签
FROM    summerfarm_tech.app_log_search_detail_di
WHERE   ds BETWEEN '{last_n_days_ago}' and '{ds_yesterday}'
GROUP BY query
ORDER BY searched_users DESC;
"""

top_query_df = get_odps_sql_result_as_df(sql=top_query)
top_query_df.head(20)

In [ ]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    query = f"""
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{{digit}}') as api,pageName as page_ame,
json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].experimentId') experiment_id,
type,uid,date_format(__time__, '%Y%m%d') as ds,count(1) search_times,
array_join(array_sort(array_agg(distinct json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].variantId'))),',') variant_list
from log group by 1,2,3,4,5,6 limit 1000000"""
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    _df["search_times"] = _df["search_times"].fillna(1).astype(int)

    _df["variant_list"] = _df["variant_list"].fillna("none")

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_variant_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print("今天的数据还未完整，跳过")
        break
    df = get_user_variant_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_variant_df.head(10)

In [ ]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,search_query,type,uid,ds
from(
select uid,date_format(__time__, '%Y%m%d') ds,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print("今天的数据还未完整，跳过")
        break
    df = get_user_sku_view_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_view_df.head(10)

In [ ]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and pageName:/search/goods |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item, 'name:([^,]+)', 1)) AS name,
  coalesce(sku,regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1)) AS sku,
  coalesce(pid,regexp_extract(sku_item, 'pid:([^,]+)', 1)) AS pid,
  coalesce(pdid,regexp_extract(sku_item, 'pdid:(\d+)', 1)) AS pdid,bid,
ds,search_query,type,uid,page_name,sku_item,coalesce(linkInfo,url)linkInfo from(
select uid,date_format(__time__, '%Y%m%d') ds,replace(replace(split_part(url_decode(split_part(url,'#/',2)),'?',2),'=',':'),'&',',') url,
bid_list.sku_item,pageName as page_name,bid,idx,name,sku,pid,pdid,linkInfo,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 1000000)"""

# click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_raw_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
            errors="ignore",
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print("今天的数据还未完整，跳过")
        break
    df = get_user_sku_click_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

In [ ]:
import re

all_user_sku_click_explored = []
pattern = re.compile(r'idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)')

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    bid = _dict["bid"]
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)
        search_query=_dict["linkInfo"]
        sku_info["search_query"] = re.search(r'pdName:([^,]+)', search_query).group(1)
        sku_info["bid"] = bid_item
        if 'undefined' in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = sku = pid = name = None
                
                idx_match = re.search(r'idx:(\d+)', bid_item)
                if idx_match:
                    idx = idx_match.group(1)
                    
                pdid_match = re.search(r'pdid:(\d+)', bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)
                    
                sku_match = re.search(r'sku:([\dA-Za-z]+)', bid_item)
                if sku_match:
                    sku = sku_match.group(1)
                    
                pid_match = re.search(r'pid:([^,]+)', bid_item)
                if pid_match:
                    pid = pid_match.group(1)
                    
                name_match = re.search(r'name:([^,]+)', bid_item)
                if name_match:
                    name = name_match.group(1)
                    
                sku_info.update({
                    "idx": idx,
                    "pdid": pdid, 
                    "sku": sku,
                    "pid": pid,
                    "name": name
                })
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[['bid','sku','name','idx','pid','pdid','linkInfo','search_query']].head(5)

In [ ]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


In [ ]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

In [ ]:
import pandasql

_df=pandasql.sqldf("select ds,action_type,count(1) cnt from user_click_with_variant_df where idx is null group by 1,2; ")
_df

In [98]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].fillna(-1)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].replace('null', -1).astype(int)

In [ ]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf(
    """
select uid,variant_list,ds,b.搜索频次标签,
count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
count(case when (action_type='唤起购买' and variant_list is not null) or action_type='商品详情' then 1 end) as 总点击cnt,
count(case when (action_type='唤起购买' and variant_list is not null or action_type='商品详情') and idx>=0 and idx<=5 then 1 end) as 首屏总点击cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
max(case when action_type='商品详情' or action_type='唤起购买' then idx end) as max点击位置,
min(case when action_type='商品详情' or action_type='唤起购买' then idx end) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,b.搜索频次标签
""")

user_click_with_variant_statistics_df.head(5)

In [100]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)
user_view_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,ds,variant_list,b.搜索频次标签,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,b.搜索频次标签
"""
)


In [ ]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list","搜索频次标签"], how="left"
)

all_data_df["商品详情cnt"] = all_data_df["商品详情cnt"].fillna(0).astype(int)
all_data_df["加入购物车cnt"] = all_data_df["加入购物车cnt"].fillna(0).astype(int)
all_data_df["唤起购买cnt"] = all_data_df["唤起购买cnt"].fillna(0).astype(int)
all_data_df["avg点击位置"] = all_data_df["avg点击位置"].fillna(0.0).astype(float)
all_data_df["sku_click_rate"] = (all_data_df['商品详情cnt']*1.00/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
all_data_df["add_cart_rate"] = (all_data_df['加入购物车cnt']/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
all_data_df["popup_click_rate"] = (all_data_df['唤起购买cnt']/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
print(all_data_df.columns)

In [102]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 90vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 2rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.5rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v2(row: pd.Series, metric: str = "diff_to_v2%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v2%"] = df_to_display.apply(display_diff_to_v2, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [105]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
    control_variant: str = "V2",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == control_variant][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4", "V5"]:
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        print(f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}')
        if len(test_group["ds"].unique())<=0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            f"diff_to_{control_variant}%".lower(): round(100.00*(test.mean() - control_avg) / control_avg,2),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(
                    test_group[test_group[metric] > 0][
                        ["uid", "ds"]
                    ].drop_duplicates()
                )
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False)
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [ ]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4", "V5"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "add_cart_rate",
    "popup_click_rate",
    "商品详情cnt",
    "加入购物车cnt",
    "唤起购买cnt",
    "搜索翻页数cnt",
    "avg点击位置",
    "min点击位置",
    "总点击cnt",
    "首屏总点击cnt",
]
all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    for label, group_df in all_data_df.groupby("搜索频次标签"):
        p_values_df = calculate_p_values(
            group_df,
            metric=metric,
        )

        print(label)

        p_values_df["搜索频次"] = label

        p_values_df["variant_list"] = p_values_df["variant_list"].apply(
            lambda x: x if x in variant_order else "X_" + x
        )
        p_values_df = p_values_df.sort_values(
            by="variant_list", key=lambda x: x.map(sort_key)
        )
        p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")

        title = (
            f"搜索AB--{metric}-{label}_p-value分布-{p_values_df.iloc[0]['日期范围']}"
        )

        html_content = dataframe_to_html(df=p_values_df, title=title)
        file_path = f"./data/{title}.html"

        # 保存HTML到本地文件：
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(html_content)

        print(f"写入HTML成功！{file_path}")
        all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)
        title_all = f"搜索AB--指标全集_p-value分布-{p_values_df.iloc[0]['日期范围']}"
        html_content = dataframe_to_html(df=all_p_values_df, title=title_all)
        file_path = f"./data/{title_all}.html"

        # 保存HTML到本地文件：
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(html_content)

        print(f"写入HTML成功！{file_path}")

all_p_values_df

In [ ]:
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

write_pandas_df_into_odps(
    df=all_user_variant_df,
    table_name="summerfarm_ds.temp_search_ab_all_data_df",
    partition_spec=partition_spec,
    overwrite=True,
    lifecycle=30,
)

# 只分析哪些进入过搜索页面的用户的订单转化结果

order_query = """
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-01-01 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_all_data_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_all_data_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

user_orders_df = get_odps_sql_result_as_df(order_query)
user_orders_df.head(2)

In [ ]:
user_orders_df["order_gmv"]=user_orders_df["order_gmv"].astype(float)
user_orders_df["avg_order_gmv"]=user_orders_df["avg_order_gmv"].astype(float)
user_orders_df["order_cnt"]=user_orders_df["order_cnt"].astype(int)
user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].describe()

In [ ]:
print(
    f"所有订单的分布:\n",
    user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].quantile(
        [0.5, 0.75, 0.95, 0.99, 0.995, 0.996, 0.997, 0.999, 1]
    ),
)


# 这里排除哪些高单价的订单，否则对于数据分析来说不好处理。
user_orders_below_6k_df = user_orders_df[user_orders_df["order_gmv"] <= 6000]
print(
    "排除高单价的订单后的分布:\n",
    user_orders_below_6k_df["order_gmv"].quantile(
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ),
)

In [ ]:
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].fillna(0)
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(float)

user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df["avg_order_gmv"].fillna(0.0)
user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df["avg_order_gmv"].astype(float)

user_orders_below_6k_df["category1"] = "ignore"
user_orders_below_6k_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_below_6k_df[
    user_orders_below_6k_df["variant_list"].isin(["V1", "V2", "V3", "V4", "V5"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)